## Week 4.2: Modules and packages.

A notebook based on the [Python tutorial](https://docs.python.org/3/tutorial), which you are invited to consult.

#### Table of contents:
**The notebook is organized as follows:**

* **[Part 1: Python scopes and namespace](#421-python-scopes-and-namespace)**

* **[Part 2: Modules](#422-modules)**

* **[Part 3: Packages](#423-packages)**


### 4.2.1 Python scopes and namespace

A **namespace** is a mapping from names to objects. Examples of namespaces include:

- the set of built-in names (functions such as `abs()`, and built-in exception names)
- the global names in a module
- the local names in a function invocation

Namespaces are created at different moments and have different lifetimes. Different namespaces can contain the same name without any collision — for instance, both module `A` and module `B` could each define a function `maxvalue` without confusion, as long as you prefix it with the module name when using it from outside (`A.maxvalue`, `B.maxvalue`).

An **attribute** is any name following a dot — e.g. in `z.real`, `real` is an attribute of the object `z`.

A **scope** is a textual region of a Python program where a namespace is directly accessible (i.e., where an unqualified reference to a name attempts to find it). At any time during execution there are (at least) three or four nested scopes whose namespaces are directly accessible, searched in this order:

1. the innermost scope, containing local names
2. the scopes of any enclosing functions, containing non-local, non-global names
3. the next-to-last scope, containing the current module's global names
4. the outermost scope, containing built-in names

This search order is often called the **LEGB rule**: **L**ocal, **E**nclosing, **G**lobal, **B**uilt-in.

In [ ]:
def scope_test():
    def do_local():
        spam = "local spam"

    def do_nonlocal():
        nonlocal spam
        spam = "nonlocal spam"

    def do_global():
        global spam
        spam = "global spam"

    spam = "test spam"
    do_local()
    print("After local assignment:", spam)
    do_nonlocal()
    print("After nonlocal assignment:", spam)
    do_global()
    print("After global assignment:", spam)

scope_test()
print("In global scope:", spam)

Notice how the *local* assignment (which is default) didn't change `scope_test`'s binding of `spam`. The `nonlocal` assignment changed `scope_test`'s binding of `spam`, and the `global` assignment changed the module-level binding.

You can also see that there was no previous binding for `spam` before the `global` assignment — it was created in the module's global namespace by that assignment.

#### Exercise 1 — Reading the LEGB rule

For each name *used* inside `inner` (`len`, `str`, `print`, `x`, `y`, `z`), decide in which scope Python finds it:
`'L'` (local), `'E'` (enclosing), `'G'` (global) or `'B'` (built-in).
Fill in the dictionary `scope_of` (run the cell first to see the code work).

In [ ]:
x = 10

def outer():
    y = 20

    def inner():
        z = 30
        print(len(str(x + y + z)))

    inner()

outer()

# TODO: replace each None by 'L', 'E', 'G' or 'B'
scope_of = {
    "len": None,
    "str": None,
    "print": None,
    "x": None,
    "y": None,
    "z": None,
}

# --- Check your answer (do not edit) ---
import hashlib, json
_digest = hashlib.sha256(json.dumps(scope_of, sort_keys=True).encode()).hexdigest()[:10]
assert _digest == "0ff8980416", "Not quite. Remember the order: Local, Enclosing, Global, Built-in."
print("Correct!")

#### The `global` statement

The `global` statement is used to indicate that particular variables live in the global (module) scope and should be rebound there.

In [ ]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
increment()
print(counter)

#### Exercise 2 — The `global` statement

The function below tries to add to a module-level running total, but something is wrong.
Run the cell (the `try/except` just prints the error instead of stopping the notebook).

In [ ]:
running_total = 0

def broken_add(n):
    running_total += n

try:
    broken_add(5)
except Exception as e:
    print(type(e).__name__ + ":", e)

**(a)** Which error do you get? Why does Python talk about a *local* variable, even though the code never wrote `running_total = ...`?

**Your answer:** *(double-click this cell to write it)*

**(b)** Write a corrected version, `add_to_total(n)`, that really updates the module-level `running_total`.

In [ ]:
running_total = 0

def add_to_total(n):
    # TODO: fix this function
    running_total += n

add_to_total(5)
add_to_total(7)

# --- Check your answer (do not edit) ---
assert running_total == 12, f"expected 12, got {running_total}"
print("Correct!")

**(c)** The next function modifies a global list and works fine **without** any `global` statement.

In [ ]:
log = []

def record(msg):
    log.append(msg)

record("hello")
print(log)

Why is `global` not needed in `record`, while it was needed in `add_to_total`?

**Your answer:** *(double-click this cell to write it)*

#### The `nonlocal` statement

The `nonlocal` statement is used inside nested functions to indicate that a particular variable lives in an enclosing scope (but not the global scope) and should be rebound there. This is the classic way to build a *closure*.

In [ ]:
counter = 0
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

make_counter()
make_counter()
print(counter)

c = make_counter()
print(c())
print(c())
print(c())

#### Exercise 3 — The `nonlocal` statement

Write a function `make_accumulator(start=0)` that returns a function `add(n)`.
Each call `add(n)` adds `n` to a running total (initially `start`) and returns the new total.
Two accumulators created separately must not share their totals.

In [ ]:
def make_accumulator(start=0):
    # TODO
    ...

# --- Check your answer (do not edit) ---
acc1 = make_accumulator()
acc2 = make_accumulator(100)
assert acc1(5) == 5
assert acc1(10) == 15
assert acc2(1) == 101
assert acc1(1) == 16, "accumulators must be independent"
print("Correct!")

#### Exercise 4 — Three different `x`

**(a)** Predict what the following code prints, *then* run the cell to check your answer.

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer()
print("module:", x)

Write down your prediction and explain it in one or two sentences.

**Your answer:** *(double-click this cell to write it)*

**(b)** Add **one** statement at the marked place so that `inner` rebinds the `x` of the *enclosing* function `outer_b`.
Expected output:
```
inner: local
outer: local
module: global
```

In [ ]:
x = "global"

def outer_b():
    x = "enclosing"

    def inner():
        # TODO: add ONE statement here
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer_b()
print("module:", x)

# --- Check your answer (do not edit) ---
import io, contextlib
x = "global"
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    outer_b()
    print("module:", x)
assert buf.getvalue().splitlines() == ["inner: local", "outer: local", "module: global"], buf.getvalue()
print("Correct!")

**(c)** Now add **one** statement so that `inner` rebinds the *module-level* `x` instead. Expected output:
```
inner: local
outer: enclosing
module: local
```

In [ ]:
x = "global"

def outer_c():
    x = "enclosing"

    def inner():
        # TODO: add ONE statement here
        x = "local"
        print("inner:", x)

    inner()
    print("outer:", x)

outer_c()
print("module:", x)

# --- Check your answer (do not edit) ---
import io, contextlib
x = "global"
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    outer_c()
    print("module:", x)
assert buf.getvalue().splitlines() == ["inner: local", "outer: enclosing", "module: local"], buf.getvalue()
x = "global"   # restore
print("Correct!")

#### `dir()` and namespaces

The built-in function `dir()` is used to find out which names a module (or the current scope) defines. Without arguments, `dir()` lists the names you have defined in the current scope.

In [ ]:
a_variable = 42
def a_function():
    pass

names = dir()
[n for n in names if not n.startswith('__')]  #Names starting with __ are Python's internal/special names

Note that `dir()` does **not** list the names of built-in functions and variables by default. If you want that list, they are defined in the standard module `builtins`:

In [ ]:
import builtins
len(dir(builtins))

#### Exercise 5 — `dir()` and namespaces

**(a)** Store in `result_a` a boolean that is `True` when `'abs'` is one of the built-in names but is **not** among the names of the current namespace.

In [ ]:
import builtins

result_a = ...   # TODO

# --- Check your answer (do not edit) ---
assert result_a is True
print("Correct!")

**(b)** Inside a function, `dir()` without argument lists the names of the *local* namespace. Complete `local_names` so that it returns them.

In [ ]:
def local_names(a, b):
    c = a + b
    # TODO: return the sorted list of the names local to this function, using dir()

# --- Check your answer (do not edit) ---
assert local_names(1, 2) == ['a', 'b', 'c']
print("Correct!")

**(c)** Predict what happens at the first `print`, then finish the cell: add **one** statement that removes the shadowing so that the last line works.

In [ ]:
max = 10

try:
    print(max(3, 7))
except TypeError as e:
    print("TypeError:", e)

# TODO: undo the shadowing (one statement)

print(max(3, 7))

# --- Check your answer (do not edit) ---
assert max(3, 7) == 7
print("Correct!")

In which scope did Python find `max` at the first `print`? What does this show about the LEGB order?

**Your answer:** *(double-click this cell to write it)*

### 4.2.2 Modules

As your programs get longer, you'll want to split them into several files for easier maintenance. A **module** is a file containing Python definitions and statements. The file name is the module name with the suffix `.py` appended.

Within a module, the module's name (as a string) is available as the value of the global variable `__name__`.

For example, suppose you have a file `fibo.py` in the current directory with the following contents:

```python
# Fibonacci numbers module

def fib(n):    # write Fibonacci series up to n
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a+b
    print()

def fib2(n):   # return Fibonacci series up to n
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a+b
    return result
```

You would then enter the Python interpreter and import this module with `import fibo`. This does not enter the names of the functions defined in `fibo` directly in the current namespace — it only enters the module name `fibo` there. Using the module name you can access the functions: `fibo.fib(1000)`.

Let's create this module on disk and explore it for real.

In [ ]:
fibo_source = '''
# Fibonacci numbers module

def fib(n):    # write Fibonacci series up to n
    a, b = 0, 1
    while a < n:
        print(a, end=' ')
        a, b = b, a + b
    print()

def fib2(n):   # return Fibonacci series up to n
    result = []
    a, b = 0, 1
    while a < n:
        result.append(a)
        a, b = b, a + b
    return result
'''

with open('fibo.py', 'w') as f:
    f.write(fibo_source)

print('fibo.py written to disk')

In [ ]:
import fibo

fibo.fib(1000)

In [ ]:
fibo.fib2(100)

In [ ]:
fibo.__name__

If you intend to use a function often, you can assign it to a local name:

In [ ]:
fib = fibo.fib
fib(500)

#### Exercise 2.1 — Using a module

Using the `fibo` module imported above:

* **(a)** store in `fib_sum` the sum of all Fibonacci numbers strictly below 1000 (use `fibo.fib2`);
* **(b)** write `count_fib(n)`, which returns how many Fibonacci numbers are strictly less than `n`, again relying on `fibo`.

In [ ]:
# (a)
fib_sum = ...

# (b)
def count_fib(n):
    # Return how many Fibonacci numbers are strictly less than n
    ...

# --- Check your answer (do not edit) ---
assert fib_sum == 2583, f"fib_sum is {fib_sum}"
assert count_fib(100) == 12
assert count_fib(1) == 1
assert count_fib(2) == 3
print("Correct!")

**(c)** Run the cell below.

In [ ]:
print("this notebook:", __name__)
print("the fibo module:", fibo.__name__)

Why is the value of `__name__` different in the two cases?

**Your answer:** *(double-click this cell to write it)*

#### More on modules

A module can contain executable statements as well as function definitions. These statements are intended to initialize the module and are executed only the **first time** the module name is encountered in an import statement (they are also run if the file is executed as a script).

Each module has its own private namespace, used as the global namespace by all functions defined in the module. This means the author of a module can use global variables without worrying about accidental clashes with a user's global variables.

Modules can import other modules. It is customary (but not required) to place all `import` statements at the beginning of a module.

There is a variant of the `import` statement that imports names from a module directly into the importing module's namespace:

In [ ]:
from fibo import fib, fib2
fib(200)

There is even a variant to import all names that a module defines, using `from fibo import *`. This imports all names except those beginning with an underscore. **In general, this practice is discouraged** since it introduces an unknown set of names into the interpreter, possibly hiding some things you have already defined, and makes it much harder for readers to know where a name came from.

If the module name is followed by `as`, then the name following `as` is bound directly to the imported module:

In [ ]:
import fibo as fib_module
fib_module.fib(30)

In [ ]:
from fibo import fib as fibonacci
fibonacci(50)

#### Exercise 2.2 — The different forms of `import`

The cell below creates a small module `mymath.py` and a helper `star_import(package)` that runs `from package import *` in a scratch namespace and returns the (sorted) names that arrive. Just run it.

In [ ]:
mymath_source = '''
PI = 3.14159

def area(r):
    return PI * r * r

def perimeter(r):
    return 2 * PI * r

def _secret():
    return 42
'''

with open('mymath.py', 'w') as f:
    f.write(mymath_source)

import importlib
importlib.invalidate_caches()

def star_import(package):
    # Run `from <package> import *` in a scratch namespace; return the sorted names that arrive
    ns = {}
    exec(f"from {package} import *", ns)
    return sorted(n for n in ns if not n.startswith("__"))

* **(a)** Import the module under the alias `mm` and store in `area_2` the area of a circle of radius 2.
* **(b)** Import **only** the function `perimeter`, under the name `circ`, and store in `perim_1` the perimeter of a circle of radius 1.
  (After (a) and (b), the names `mymath` and `perimeter` must **not** exist in the notebook namespace.)

In [ ]:
# (a)
...   # TODO: import
area_2 = ...

# (b)
...   # TODO: import
perim_1 = ...

# --- Check your answer (do not edit) ---
assert abs(area_2 - 12.56636) < 1e-9
assert abs(perim_1 - 6.28318) < 1e-9
assert "mymath" not in globals() and "perimeter" not in globals()
print("Correct!")

**(c)** Which names does `from mymath import *` bring in? Give your prediction as a list sorted alphabetically (is `_secret` in it?).

In [ ]:
prediction_c = None   # TODO: a sorted list of names

# --- Check your answer (do not edit) ---
actual_c = star_import("mymath")
assert prediction_c == actual_c, "Not quite: re-read the rule about names starting with an underscore."
print("Correct!")

#### Executing modules as scripts

When you run a Python module with:

```bash
python fibo.py <arguments>
```

the code in the module will be executed, just as if you imported it, **but with `__name__` set to `"__main__"`**. By adding this code at the end of your module:

```python
if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]))
```

you can make the file usable both as a script and as an importable module. This is an extremely common idiom — it's a way to write code that only runs when the file is executed directly, not when it's imported elsewhere.

In [ ]:
script_addition = '''

if __name__ == "__main__":
    import sys
    fib(int(sys.argv[1]) if len(sys.argv) > 1 else 100)
'''

with open('fibo.py', 'a') as f:
    f.write(script_addition)

print('Updated fibo.py:')
with open('fibo.py') as f:
    print(f.read())

In [ ]:
import subprocess
result = subprocess.run(['python3', 'fibo.py', '300'], capture_output=True, text=True)
print(result.stdout)

#### Exercise 2.3 — Executing modules as scripts

Write (as a string, then to disk) a module `greeter.py` with:

* a function `greet(name)` returning the string `"Hello, <name>!"`;
* a guard `if __name__ == "__main__":` such that running `python greeter.py Ada` prints `Hello, Ada!`, and running `python greeter.py` (no argument) prints `Hello, world!`.

Importing the module must **not** print anything.

In [ ]:
greeter_source = '''
import sys

def greet(name):
    # TODO
    ...

# TODO: the __main__ guard
'''

with open('greeter.py', 'w') as f:
    f.write(greeter_source)

# --- Check your answer (do not edit) ---
import contextlib, importlib, io, subprocess, sys
importlib.invalidate_caches()
sys.modules.pop("greeter", None)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    import greeter
assert buf.getvalue() == "", "importing the module must not print anything"
assert greeter.greet("Ada") == "Hello, Ada!"

out = subprocess.run([sys.executable, "greeter.py", "Ada"], capture_output=True, text=True).stdout.strip()
assert out == "Hello, Ada!", repr(out)
out = subprocess.run([sys.executable, "greeter.py"], capture_output=True, text=True).stdout.strip()
assert out == "Hello, world!", repr(out)
print("Correct!")

#### The Module Search Path

When a module named `spam` is imported, the interpreter first searches for a built-in module with that name. If not found, it searches for a file named `spam.py` in a list of directories given by the variable `sys.path`. `sys.path` is initialized from these locations:

- the directory containing the input script (or the current directory when no file is specified)
- `PYTHONPATH` (a list of directory names, with the same syntax as the shell variable `PATH`)
- the installation-dependent default (including the `site-packages` directory, handled by the `site` module)

In [ ]:
import sys
sys.path[:5]  # first few entries of the module search path

#### Standard modules

Python comes with a library of standard modules, described in the Python Library Reference. Some modules are built into the interpreter; these provide access to operations that are not part of the core of the language but are nevertheless built in, either for efficiency or to provide access to operating system primitives such as system calls. One example is the `sys` module.

In [ ]:
import sys
print("Python version:", sys.version.split()[0])
print("Platform:", sys.platform)

#### Exercise 2.4 — Standard modules

**(a)** Some modules are compiled into the interpreter itself (like `sys`), others are loaded from a file. Using `sys.builtin_module_names`, fill in the dictionary (`True` = built-in module).

**(b)** Using only standard modules (hint: `re` and `collections.Counter`), write `top_word(text)` which returns the most frequent word of `text`, in lower case, ignoring punctuation.

In [ ]:
import sys, json

# (a)
is_builtin = {
    "sys": ...,
    "fibo": ...,
    "json": ...,
}

# (b)
def top_word(text):
    ...

# --- Check your answer (do not edit) ---
assert is_builtin == {"sys": True, "fibo": False, "json": False}, is_builtin
assert top_word("The cat and the hat. The end!") == "the"
assert top_word("Red fish, blue fish, red fish!") == "fish"
print("Correct!")

### 4.2.3 Packages

Packages are a way of structuring Python's module namespace by using "dotted module names". For example, the module name `A.B` designates a submodule named `B` in a package named `A`. Just as the use of modules saves the authors of different modules from having to worry about each other's global variable names, the use of dotted module names saves the authors of multi-module packages from having to worry about each other's module names.

Suppose you want to design a collection of modules (a "package") for the uniform handling of sound files and sound data. There are many different sound file formats, so you may need to create and maintain a growing collection of modules for conversion between formats. There are also many operations you might want to perform on sound data, so you will be writing modules for those too. A possible structure for your package (expressed in terms of a hierarchical filesystem) might look like this:

```
sound/                          Top-level package
      __init__.py                Initialize the sound package
      formats/                   Subpackage for file format conversions
              __init__.py
              wavread.py
              wavwrite.py
              aiffread.py
              aiffwrite.py
              ...
      effects/                   Subpackage for sound effects
              __init__.py
              echo.py
              surround.py
              reverse.py
              ...
      filters/                   Subpackage for filters
              __init__.py
              equalizer.py
              vocoder.py
              karaoke.py
              ...
```

The `__init__.py` files are required to make Python treat directories containing the file as (regular) packages, preventing directories with a common name, such as `string`, from unintentionally hiding valid modules that occur later on the module search path. In the simplest case, `__init__.py` can just be an empty file, but it can also execute initialization code for the package or set the `__all__` variable (described later).

Let's build a small toy package on disk and actually import from it.

In [ ]:
import os

# Build the directory tree
os.makedirs('sound/formats', exist_ok=True)
os.makedirs('sound/effects', exist_ok=True)
os.makedirs('sound/filters', exist_ok=True)

files = {
    'sound/__init__.py': "'''The sound package: tools for handling sound files and data.'''\n",
    'sound/formats/__init__.py': "'''Subpackage for file format conversions.'''\n",
    'sound/formats/wavread.py': (
        'def read(path):\n'
        '    print(f"[wavread] pretending to read {path}")\n'
        '    return b"fake-wav-data"\n'
    ),
    'sound/formats/wavwrite.py': (
        'def write(path, data):\n'
        '    print(f"[wavwrite] pretending to write {len(data)} bytes to {path}")\n'
    ),
    'sound/effects/__init__.py': "'''Subpackage for sound effects.'''\n",
    'sound/effects/echo.py': (
        'def echofilter(data, delay=0.1, atten=0.4):\n'
        '    print(f"[echo] applying echo (delay={delay}, atten={atten}) to {data!r}")\n'
        '    return data\n'
    ),
    'sound/effects/surround.py': (
        'def surround(data, channels=5):\n'
        '    print(f"[surround] spreading {data!r} across {channels} channels")\n'
        '    return data\n'
    ),
    'sound/filters/__init__.py': "'''Subpackage for filters.'''\n",
    'sound/filters/equalizer.py': (
        'def equalize(data, preset="flat"):\n'
        '    print(f"[equalizer] applying {preset!r} preset to {data!r}")\n'
        '    return data\n'
    ),
}

for path, content in files.items():
    with open(path, 'w') as f:
        f.write(content)

print("Package tree created:")
for root, dirs, filenames in os.walk('sound'):
    level = root.replace('sound', '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'sound'}/")
    for fn in filenames:
        print(f"{indent}  {fn}")

Let's see what that last part is actually doing:

In [ ]:
# Walk through the 'sound' directory and all of its subdirectories.
# On each iteration:
#   root      = the directory we're currently looking at
#   dirs      = the subdirectories inside it
#   filenames = the files inside it
for root, dirs, filenames in os.walk('sound'):

    # Work out how deep we are inside the 'sound' directory.
    # For example:
    #   'sound'         -> level 0
    #   'sound/formats' -> level 1
    level = root.replace('sound', '').count(os.sep)

    # Create indentation based on how deep we are.
    # Each level gets two spaces.
    indent = '  ' * level

    # Print the name of the current directory.
    # os.path.basename(root) gets just the directory's name.
    # For example, 'sound/formats' becomes 'formats'.
    print(f"{indent}{os.path.basename(root) or 'sound'}/")

    # Go through every file in the current directory.
    for fn in filenames:

        # Print the filename with a little extra indentation
        # so it appears underneath its directory.
        print(f"{indent}  {fn}")

#### Exercise 3.1 — Building a package on disk

First, a small helper used by several exercises of this part: `forget(prefix)` removes a package and its submodules from `sys.modules` (it does what the loops in the cells below do), so that the next `import` starts from scratch. Just run it.

In [ ]:
import os, sys, importlib

def forget(prefix):
    # Remove `prefix` and all its submodules from sys.modules, forcing a fresh import next time
    for name in [m for m in sys.modules if m == prefix or m.startswith(prefix + ".")]:
        del sys.modules[name]
    importlib.invalidate_caches()

**(a)** Add a new subpackage `sound/analysis` to the package built above. It must contain:

* an `__init__.py` (a one-line docstring is enough);
* a module `spectrum.py` defining `peak_frequency(data)`, which simply returns `440`.

In [ ]:
os.makedirs("sound/analysis", exist_ok=True)
...   # TODO: write the two files

# --- Check your answer (do not edit) ---
assert os.path.isfile("sound/analysis/__init__.py")
assert os.path.isfile("sound/analysis/spectrum.py")
assert "def peak_frequency" in open("sound/analysis/spectrum.py").read()
print("Correct!")

**(b)** The next cell creates a directory `nsdemo/` with a module `utils.py` but **deliberately without an `__init__.py`**.
Predict whether `import nsdemo.utils` works (`True`/`False`), then run the cell.

In [ ]:
prediction_works = None   # TODO: True or False

os.makedirs("nsdemo", exist_ok=True)
with open("nsdemo/utils.py", "w") as f:
    f.write("def hello():\n    return 'hi'\n")
importlib.invalidate_caches()

try:
    import nsdemo.utils
    works = True
except ImportError:
    works = False
print("works:", works)
if works:
    print("nsdemo.__file__ :", getattr(nsdemo, "__file__", None))
    print("nsdemo.__path__ :", nsdemo.__path__)

# --- Check your answer (do not edit) ---
assert prediction_works == works
print("Correct!")

`nsdemo` was found although it has no `__init__.py`. What kind of package is it? Given this, why do we still put an `__init__.py` in our packages?

**Your answer:** *(double-click this cell to write it)*

#### Importing from packages

Users of the package can import individual modules from the package, for example:

```python
import sound.effects.echo
```

This loads the submodule `sound.effects.echo`. It must be referenced with its full name, e.g. `sound.effects.echo.echofilter(...)`.

An alternative way of importing the submodule is:

```python
from sound.effects import echo
```

This also loads the submodule `echo`, and makes it available *without* the package prefix, so it can be used as `echo.echofilter(...)`.

Yet another variation is to import the desired function or variable directly:

```python
from sound.effects.echo import echofilter
```

This loads the submodule `echo` again, but makes its function `echofilter()` directly available: `echofilter(...)`.

Let's try each style.

In [ ]:
import sound.effects.echo

sound.effects.echo.echofilter(b"raw-audio", delay=0.2, atten=0.5)

In [ ]:
from sound.effects import surround

surround.surround(b"raw-audio", channels=7)

In [ ]:
from sound.filters.equalizer import equalize

equalize(b"raw-audio", preset="bass-boost")

Note that when using `from package import item`, the item can be either a submodule (or subpackage) of the package, or some other name defined in the package, like a function, class or variable. The `import` statement first tests whether the item is defined in the package; if not, it assumes it is a module and attempts to load it. If it fails to find it, an `ImportError` exception is raised.

Contrarily, when using syntax like `import item.subitem.subsubitem`, each item except for the last must be a package; the last item can be a module or a package, but cannot be a class, function or variable defined in the previous item.

#### Exercise 3.2 — Importing from a package

Use the `sound` package to complete each item **with the import style requested**:

* **(a)** with the full dotted name: `import sound.effects.echo`, then call `echofilter(b"x")`;
* **(b)** with `from sound.effects import ...`, binding the submodule to the name `echo_mod`, then call its `echofilter(b"x")`;
* **(c)** import `read` from `sound.formats.wavread` under the name `read_wav`, and read `"song.wav"`;
* **(d)** import the module `spectrum` of the subpackage created in Exercise 3.1 in the style you prefer, and call `peak_frequency(b"x")`.

In [ ]:
# (a)
import sound.effects.echo
out_a = ...

# (b)
...
out_b = ...

# (c)
...
out_c = ...

# (d)
...
out_d = ...

# --- Check your answer (do not edit) ---
assert out_a == b"x" and out_b == b"x"
assert out_c == b"fake-wav-data"
assert out_d == 440
print("Correct!")

**(e)** For each statement below, predict the outcome: `"OK"`, `"ImportError"` or `"ModuleNotFoundError"`
(the exact class name; `ModuleNotFoundError` is a subclass of `ImportError`). Re-read the note above about `from package import item` versus `import item.subitem`.

In [ ]:
prediction_e = {
    "import sound.effects.echo.echofilter": None,
    "from sound.effects.echo import echofilter": None,
    "from sound.effects import nonexistent": None,
}

# --- Check your answer (do not edit) ---
actual_e = {}
for stmt in prediction_e:
    try:
        exec(stmt, {})
        actual_e[stmt] = "OK"
    except Exception as e:
        actual_e[stmt] = type(e).__name__
assert prediction_e == actual_e, "Not quite: in `import a.b.c` every item but the last must be a package."
print("Correct!")

#### Importing `*` from a package

The only solution is for the package author to provide an explicit index of the package. The `import` statement uses the following convention: if a package's `__init__.py` code defines a list named `__all__`, it is taken to be the list of module names that should be imported when `from package import *` is encountered.

Let's give the `effects` subpackage an `__all__` list and see the difference.

In [ ]:
with open('sound/effects/__init__.py', 'w') as f:
    f.write(
        "'''Subpackage for sound effects.'''\n"
        '__all__ = ["echo", "surround"]\n'
    )

# Force a clean reimport of everything under sound.effects
import sys
for mod_name in list(sys.modules):
    if mod_name.startswith('sound'):
        del sys.modules[mod_name]

from sound.effects import *

# Only names listed in __all__ (echo, surround) are now bound here
print("echo module available:", 'echo' in dir())
print("surround module available:", 'surround' in dir())
print("reverse module available:", 'reverse' in dir())  # not in __all__, and doesn't even exist on disk

If `__all__` is not defined, `from sound.effects import *` does **not** import all submodules from the package `sound.effects` into the current namespace — it only ensures that the package `sound.effects` has been imported (running any initialization code in `__init__.py`) and then imports whatever names are defined in the package's namespace, including any submodules explicitly loaded by previous `import` statements.

Remember, using `from package import *` is generally considered bad practice in production code, since it can lead to unreadable code and unknown/shadowed names in a program.

#### Exercise 3.3 — `from package import *` and `__all__`

The cell below creates a package `toolbox` with three modules `a`, `b`, `c` (each defining `NAME`) and an **empty** `__init__.py`.

In [ ]:
os.makedirs("toolbox", exist_ok=True)
for name in ["a", "b", "c"]:
    with open(f"toolbox/{name}.py", "w") as f:
        f.write(f'NAME = "{name}"\n')
with open("toolbox/__init__.py", "w") as f:
    f.write("")

def star_import(package):
    # Run `from <package> import *` in a scratch namespace; return the sorted names that arrive
    ns = {}
    exec(f"from {package} import *", ns)
    return sorted(n for n in ns if not n.startswith("__"))

**(a)** Predict the list of names brought in by `from toolbox import *` right now (nothing has imported `a`, `b` or `c` yet).

In [ ]:
prediction_a = None   # TODO: a sorted list of names

# --- Check your answer (do not edit) ---
assert prediction_a == star_import("toolbox"), "Not quite: without __all__, submodules are not imported by *"
print("Correct!")

**(b)** Rewrite `toolbox/__init__.py` so that `from toolbox import *` brings in only `a` and `b`, not `c`.

In [ ]:
...   # TODO: write toolbox/__init__.py

forget("toolbox")
result_b = star_import("toolbox")
print(result_b)

# --- Check your answer (do not edit) ---
assert result_b == ["a", "b"]
print("Correct!")

**(c)** `c` is not in `__all__`. Can you still import it explicitly? Do it, and store `c.NAME` in `c_name`.

In [ ]:
...
c_name = ...

# --- Check your answer (do not edit) ---
assert c_name == "c"
print("Correct!")

#### Intra-package references

When packages are structured into subpackages, you can use *absolute imports* to refer to submodules of sibling packages. For example, if the module `sound.filters.vocoder` needs to use the `echo` module in `sound.effects`, it can use `from sound.effects import echo`.

You can also write *relative imports*, with the `from module import name` form of import statement. These imports use leading dots to indicate the current and parent packages involved in the relative import. From the `sound.effects.surround` module, for example, you might use:

```python
from . import echo          # sibling module in the same subpackage
from .. import formats      # subpackage of the parent package
from ..filters import equalizer  # sibling subpackage of the parent package
```

Note that relative imports are based on the name of the *current module*. Since the name of the main module is always `"__main__"`, modules intended for use as the main module of a Python application must always use absolute imports.

Let's demonstrate a relative import by editing `surround.py` to use `echo` via a relative import, then run it as part of the package (not as a top-level script, since relative imports require the module to be part of a package).

In [ ]:
with open('sound/effects/surround.py', 'w') as f:
    f.write(
        'from . import echo\n'
        '\n'
        'def surround(data, channels=5):\n'
        '    data = echo.echofilter(data, delay=0.05, atten=0.6)\n'
        '    print(f"[surround] spreading {data!r} across {channels} channels")\n'
        '    return data\n'
    )

# Clean re-import
import sys
for mod_name in list(sys.modules):
    if mod_name.startswith('sound'):
        del sys.modules[mod_name]

from sound.effects.surround import surround
surround(b"raw-audio", channels=7)

#### Exercise 3.4 — References inside a package

**(a)** Create the module `sound/analysis/report.py` using **relative imports** only:

* it imports its sibling module `spectrum`;
* it imports `equalizer` from the sibling subpackage `filters`;
* it defines `analyze(data)`, which equalizes `data` (default preset) and returns the peak frequency of the result;
* when run as the main program it prints `analyze(b"demo")`.

In [ ]:
report_source = '''
# TODO: the two relative imports

def analyze(data):
    ...

if __name__ == "__main__":
    print(analyze(b"demo"))
'''

with open("sound/analysis/report.py", "w") as f:
    f.write(report_source)

# --- Check your answer (do not edit) ---
forget("sound")
from sound.analysis.report import analyze
assert analyze(b"demo") == 440
assert "from ." in report_source, "use RELATIVE imports here"
print("Correct!")

**(b)** Now run this module in two different ways, and look at the results.

In [ ]:
import subprocess

# 1) as a plain script, given its file path
direct = subprocess.run([sys.executable, "sound/analysis/report.py"], capture_output=True, text=True)
print("direct  -> exit code", direct.returncode, "|", direct.stderr.strip().splitlines()[-1])

# 2) as a module of the package
as_module = subprocess.run([sys.executable, "-m", "sound.analysis.report"], capture_output=True, text=True)
print("with -m -> exit code", as_module.returncode, "| last line:", as_module.stdout.strip().splitlines()[-1])

Why does the first way fail while the second one works?

**Your answer:** *(double-click this cell to write it)*

**(c)** Create `sound/analysis/report_abs.py`: the same module, but using **absolute imports** (`sound.analysis...`, `sound.filters...`).

In [ ]:
report_abs_source = '''
# TODO
'''

with open("sound/analysis/report_abs.py", "w") as f:
    f.write(report_abs_source)

# --- Check your answer (do not edit) ---
forget("sound")
from sound.analysis.report_abs import analyze as analyze_abs
assert analyze_abs(b"demo") == 440
assert "from sound" in report_abs_source and "from ." not in report_abs_source
print("Correct!")

#### Packages in multiple directories

Packages support one more special attribute, `__path__`. This is initialized to be a list containing the name of the directory holding the package's `__init__.py` before the code in that file is executed. This variable can be modified; doing so affects future searches for modules and subpackages contained in the package. While this feature is not often needed, it can be used to extend the set of modules found in a package.

In [ ]:
import sound
sound.__path__

#### Exercise 3.5 — The `__path__` attribute

**(a)** Store in `effects_dirs` the list of directories in which the package `sound.effects` looks for its submodules.

**(b)** The cell creates a directory `extra_effects/`, *outside* the package, containing a module `reverse.py`.
Extend `sound.effects.__path__` so that `from sound.effects import reverse` works.

In [ ]:
import os, importlib
import sound.effects

# (a)
effects_dirs = ...

# (b)
extra_effects = os.path.abspath("extra_effects")
os.makedirs(extra_effects, exist_ok=True)
with open(os.path.join(extra_effects, "reverse.py"), "w") as f:
    f.write("def reverse(data):\n    return data[::-1]\n")
importlib.invalidate_caches()

...   # TODO: extend the package's __path__

from sound.effects import reverse

# --- Check your answer (do not edit) ---
assert effects_dirs[0].endswith(os.path.join("sound", "effects"))
assert reverse.reverse(b"abc") == b"cba"
print("Correct!")

#### Exercise 3.6 — Another submodule in `__all__`

1. Add a `filters/karaoke.py` module with a function `remove_vocals(data)` that just prints a message and returns `data`.
2. Give `sound/filters/__init__.py` an `__all__` list containing `"equalizer"` and `"karaoke"`.
3. Import everything from `sound.filters` with `from sound.filters import *` and call `karaoke.remove_vocals(...)`.

In [ ]:
# 1. sound/filters/karaoke.py
...

# 2. sound/filters/__init__.py
...

# 3. clean re-import, then use it
forget("sound")
from sound.filters import *
karaoke.remove_vocals(b"raw-audio")

# --- Check your answer (do not edit) ---
assert karaoke.remove_vocals(b"raw") == b"raw"
assert star_import("sound.filters") == ["equalizer", "karaoke"]
print("Correct!")

#### Exercise 3.7 — Challenge: your own package

Build a package `textkit` from scratch, with this structure:

```
textkit/
    __init__.py    re-exports normalize and word_count (relative imports) and defines __all__
    cleaning.py    normalize(text): lower-case, and collapse all whitespace runs into single spaces (no leading/trailing space)
    stats.py       word_count(text): number of words of normalize(text); uses normalize through a RELATIVE import
```

so that `from textkit import word_count` works and `from textkit import *` brings in exactly `normalize` and `word_count`.

In [ ]:
os.makedirs("textkit", exist_ok=True)

files = {
    "textkit/cleaning.py": '''
def normalize(text):
    ...
''',
    "textkit/stats.py": '''
# TODO: relative import of normalize

def word_count(text):
    ...
''',
    "textkit/__init__.py": '''
# TODO: re-export normalize and word_count; define __all__
''',
}

for path, content in files.items():
    with open(path, "w") as f:
        f.write(content)

# --- Check your answer (do not edit) ---
forget("textkit")
from textkit import normalize, word_count
assert normalize("  Hello   WORLD \n") == "hello world"
assert word_count("  Hello   hello  World ") == 3
assert word_count("") == 0
import textkit
assert sorted(textkit.__all__) == ["normalize", "word_count"]
assert star_import("textkit") == ["normalize", "word_count"]
print("Correct!")

#### Cleaning up

Let's remove the files we created on disk during this notebook so it leaves no clutter behind.

In [ ]:
import shutil, os

for f in ['fibo.py', 'mymath.py', 'greeter.py', 'counter_mod.py']:
    if os.path.exists(f):
        os.remove(f)

for d in ['sound', 'extra_modules', 'nsdemo', 'toolbox', 'extra_effects', 'textkit', '__pycache__']:
    if os.path.exists(d):
        shutil.rmtree(d)

print("Cleaned up.")